# Unified Weighted Domination Framework: reproducible notebook (v5)
Reproduces the manuscript results from the companion Excel workbook `PPIN_reproducibility_data.xlsx`. Keep the workbook in the same folder as this notebook. Run top to bottom. Part A: breast-cancer PPIN. Part B: synthetic dog example. The notebook regenerates the PPIN overview, MWDS visualization, Figure 2 schedule collage, and Figure 7 ranking comparison.


In [1]:
import random, itertools, time
import networkx as nx, pandas as pd, numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix
from scipy.stats import hypergeom, fisher_exact
import os
DATA='data'

## A1. Build the graph (522 nodes, 1629 distinct edges)

In [2]:
DATA_FILE='PPIN_reproducibility_data.xlsx'
metrics=pd.read_excel(DATA_FILE,sheet_name='Network_Metrics')
nodes=metrics['name'].dropna().astype(str).tolist()        # file order (used for seeds / stable sorts)
raw=pd.read_excel(DATA_FILE,sheet_name='Interaction_Edges',header=0)
G=nx.Graph(); G.add_nodes_from(nodes)
G.add_edges_from((str(a),str(b)) for a,b in zip(raw['source'],raw['target']) if a!=b)
print(G.number_of_nodes(),'nodes;',G.number_of_edges(),'distinct edges;',len(raw),'raw rows;',
      sorted(len(c) for c in nx.connected_components(G)),'component sizes')

522 nodes; 1629 distinct edges; 1654 raw rows; [2, 2, 4, 10, 504] component sizes


## A2. Weights, Eq. (1), computed on the WHOLE graph (not per component)
`w(v) = 1.05 + 0.5*NC' - 0.3*D' - 0.2*BC'` with min-max normalisation.

In [3]:
def make_weights(G,a=0.5,bd=-0.3,bb=-0.2,c0=1.05):
    deg=dict(G.degree())
    NC={v:(np.mean([deg[u] for u in G[v]]) if deg[v] else 0.0) for v in G}   # neighbourhood connectivity
    BC=nx.betweenness_centrality(G,normalized=True)
    mm=lambda d:{k:(x-min(d.values()))/(max(d.values())-min(d.values())) for k,x in d.items()}
    n,d,b=mm(NC),mm(deg),mm(BC)
    return {v:c0+a*n[v]+bd*d[v]+bb*b[v] for v in G}, deg
w,deg=make_weights(G)
print('weight range [%.4f, %.4f]'%(min(w.values()),max(w.values())))

weight range [0.6567, 1.5500]


## A3. Algorithm 2 (reverse greedy). Screening rule as implemented: `min_{u in N(v)} w(u) <= w(v)`.
Tie-breaking rule used for the published list: sort by decreasing weight, ties by **decreasing name** (gives DSG2, CASP8, RPA2).

In [4]:
def dominates(G,S):
    dom=set(S)
    for s in S: dom.update(G[s])
    return len(dom)==len(G)

def reverse_greedy(G,w,order=None):
    S=list(order) if order is not None else sorted(G,key=lambda v:(round(w[v],9),v),reverse=True)
    passes=0
    while True:
        passes+=1; removed=False; i=0
        while i<len(S):
            v=S[i]; nb=[w[u] for u in G[v]]
            if nb and min(nb)<=w[v]:
                T=S[:i]+S[i+1:]
                if dominates(G,T): S=T; removed=True; continue
            i+=1
        if not removed: return S,passes
W=lambda S,ww=None:sum((ww or w)[v] for v in S)
t0=time.time(); MWDS,passes=reverse_greedy(G,w); rt=time.time()-t0
print(len(MWDS),'proteins; total weight %.4f; passes=%d; runtime %.2fs'%(W(MWDS),passes,rt)); print(sorted(MWDS))
assert dominates(G,MWDS)
top10=sorted(G,key=lambda v:-deg[v])[:10]
print('mean degree in MWDS %.1f vs network %.1f; top-10 degree proteins in MWDS: %d'%(np.mean([deg[v] for v in MWDS]),np.mean(list(deg.values())),len(set(top10)&set(MWDS))))
print('degree 1: %.0f%%, degree<=2: %.0f%%'%(100*np.mean([deg[v]==1 for v in G]),100*np.mean([deg[v]<=2 for v in G])))

39 proteins; total weight 37.9687; passes=2; runtime 0.07s
['AKT1', 'AKT2', 'AKT3', 'ARID1A', 'BRCA1', 'BRIP1', 'CASP8', 'CBFB', 'CCND3', 'CDH1', 'CDKN1B', 'CHEK2', 'CTCF', 'DSG2', 'EGFR', 'ESR1', 'EZH2', 'FANCC', 'FOXA1', 'GATA3', 'HRAS', 'ISG15', 'MLH1', 'MSH2', 'MTDH', 'PALB2', 'PIK3CA', 'RAD51C', 'RAD51D', 'RB1', 'RPA2', 'SCUBE2', 'SMARCB1', 'SMARCD1', 'SMARCE1', 'STK11', 'TP53', 'XPC', 'XRN2']
mean degree in MWDS 18.1 vs network 6.2; top-10 degree proteins in MWDS: 6
degree 1: 42%, degree<=2: 54%


## A4. Exact optimum by integer programming (0% gap)

In [5]:
idx={v:i for i,v in enumerate(nodes)}; A=lil_matrix((len(nodes),len(nodes)))
for v in nodes:
    A[idx[v],idx[v]]=1
    for u in G[v]: A[idx[v],idx[u]]=1
res=milp(c=[w[v] for v in nodes],constraints=LinearConstraint(A.tocsr(),lb=1),integrality=np.ones(len(nodes)),bounds=Bounds(0,1))
print('ILP optimum %.4f with %d proteins; heuristic %.4f; gap %.4f%%'%(res.fun,round(sum(res.x)),W(MWDS),100*(W(MWDS)-res.fun)/res.fun))

ILP optimum 37.9687 with 39 proteins; heuristic 37.9687; gap 0.0000%


## A5. Tie-breaking: 15 random orders (seeds 0-14) among equal weights

In [6]:
cnt={};sizes=[]
for s in range(15):
    o=nodes[:]; random.Random(s).shuffle(o); o=sorted(o,key=lambda v:-round(w[v],9))   # stable sort keeps the shuffle inside ties
    R,_=reverse_greedy(G,w,order=o); sizes.append(len(R))
    for v in R: cnt[v]=cnt.get(v,0)+1
print('sizes',set(sizes),'| stable core:',sum(c==15 for c in cnt.values()),'| exchangeable:',sorted((v,c) for v,c in cnt.items() if c<15))

sizes {39} | stable core: 36 | exchangeable: [('CASP8', 3), ('DSG2', 7), ('ERBB2', 8), ('FADD', 12), ('RPA2', 6), ('RPA3', 9)]


## A6. Coefficient sensitivity: each coefficient x {0.8,0.9,1.1,1.2} (12 runs)

In [7]:
base=[0.5,-0.3,-0.2]; sets=[set(MWDS)]
for i in range(3):
    for f in (0.8,0.9,1.1,1.2):
        co=base[:]; co[i]*=f; ww,_=make_weights(G,*co); sets.append(set(reverse_greedy(G,ww)[0]))
J=[len(sets[0]&s)/len(sets[0]|s) for s in sets[1:]]
core=set.intersection(*sets)
print('sizes',sorted({len(s) for s in sets}),'| min Jaccard %.3f'%min(J),'| stable core',len(core),'| sensitive:',sorted(set.union(*sets)-core))

sizes [39, 40] | min Jaccard 0.902 | stable core 37 | sensitive: ['ANXA5', 'BRCA2', 'PPP1R9B', 'SCUBE2', 'SMARCE1']


## A7. Uniform-weight ablation (w = 1), processing order must be specified
Alphabetical, 20 random orders (seeds 0-19), degree-descending, degree-ascending.

In [8]:
one={v:1 for v in G}; ab=lambda o:reverse_greedy(G,one,order=o)[0]
al=ab(sorted(nodes)); rnd=[]
for s in range(20):
    o=nodes[:]; random.Random(s).shuffle(o); rnd.append(len(ab(o)))
dd=ab(sorted(nodes,key=lambda v:-deg[v])); da=ab(sorted(nodes,key=lambda v:deg[v]))
print('alphabetical',len(al),'| random20 %d-%d median %.0f'%(min(rnd),max(rnd),np.median(rnd)),'| degree-desc',len(dd),'| degree-asc',len(da),'weight %.4f'%W(da))

alphabetical 56 | random20 40-92 median 61 | degree-desc 334 | degree-asc 39 weight 38.0145


## A8. Centrality-product ranking (Degree x Betweenness x Closeness), size-matched top-39
Two versions: (i) `Average_C` = product of the Cytoscape-exported centralities (NetworkAnalyzer computes betweenness/closeness per connected component) -- the version in the paper's Table 4; (ii) the same product recomputed on the whole 522-node graph.

In [9]:
m=metrics[metrics['name'].notna()].copy(); m['name']=m['name'].astype(str)
cent39=m.sort_values('Average_C',ascending=False,kind='stable')['name'].tolist()[:39]        # (i) paper's version
cl=nx.closeness_centrality(G); bc=nx.betweenness_centrality(G); dc=nx.degree_centrality(G)
cent39_global=sorted(G,key=lambda v:-dc[v]*bc[v]*cl[v])[:39]                                  # (ii) whole-graph version
print('proteins with betweenness exactly 0 (whole graph):',sum(bc[v]==0 for v in G))
print('overlap of the two top-39 lists:',len(set(cent39)&set(cent39_global)))

proteins with betweenness exactly 0 (whole graph): 301
overlap of the two top-39 lists: 35


In [10]:
# Recreate manuscript figures from the data and computed results.
import matplotlib.pyplot as plt
os.makedirs('paper',exist_ok=True)

# Whole-network visualization (manuscript network overview).
_layout=nx.spring_layout(G,seed=42,iterations=80)
_hub=max(G,key=lambda v:deg[v])
plt.figure(figsize=(10,8))
nx.draw_networkx_edges(G,_layout,alpha=0.12,width=0.35,edge_color='gray')
nx.draw_networkx_nodes(G,_layout,node_size=9,node_color='#b8c6d1',linewidths=0)
nx.draw_networkx_nodes(G,_layout,nodelist=[_hub],node_size=70,node_color='#f2c94c',edgecolors='black',linewidths=0.5)
plt.axis('off'); plt.tight_layout()
plt.savefig('paper/Breast cancer protein-protein interaction network.png',dpi=300,bbox_inches='tight'); plt.close()

# MWDS-highlighted PPIN (manuscript network result figure).
plt.figure(figsize=(10,8))
nx.draw_networkx_edges(G,_layout,alpha=0.12,width=0.35,edge_color='gray')
nx.draw_networkx_nodes(G,_layout,nodelist=[v for v in G if v not in MWDS],node_size=9,node_color='#b8c6d1',linewidths=0)
nx.draw_networkx_nodes(G,_layout,nodelist=MWDS,node_size=25,node_color='red',edgecolors='darkred',linewidths=0.25)
plt.axis('off'); plt.tight_layout()
plt.savefig('paper/output (1).png',dpi=300,bbox_inches='tight'); plt.close()

# Tie-aware competition ranks: tied entries share the best rank in their block.
key_proteins=['AKT1','BRCA2','BRCA1','CHEK2','EGFR','ERBB2','ESR1','MSH2','PIK3CA','RB1','TP53']
weight_rank={v:1+sum(w[u]<w[v]-1e-9 for u in G) for v in G}
centrality_product={v:dc[v]*bc[v]*cl[v] for v in G}
centrality_rank={v:1+sum(centrality_product[u]>centrality_product[v]+1e-15 for u in G) for v in G}
red=[weight_rank[v] for v in key_proteins]; blue=[centrality_rank[v] for v in key_proteins]
x=np.arange(len(key_proteins)); width=0.34
plt.figure(figsize=(12,6))
plt.bar(x-width/2,blue,width,label='Centrality-product rank (whole graph)',color='skyblue')
plt.bar(x+width/2,red,width,label='MWDS weight rank (Eq. 1)',color='red')
plt.xticks(x,key_proteins,rotation=45,ha='right'); plt.ylabel('Rank order (lower = higher priority; ties share best rank)')
plt.xlabel('Protein (key list)'); plt.title('Comparison of protein rank: MWDS weight vs. whole-graph centrality product')
plt.legend(); plt.tight_layout(); plt.savefig('paper/2.png',dpi=300,bbox_inches='tight'); plt.close()
print('Generated PPIN overview, MWDS visualization, and Figure 7 in paper/.')


Generated PPIN overview, MWDS visualization, and Figure 7 in paper/.


## A9. Driver-gene enrichment (needs the reference list)
The companion Excel workbook contains the 299 pan-cancer driver genes of Bailey et al. (2018) on the `Driver_Genes` sheet. The MCDS (50 proteins, from Cytoscape) is listed below. Keep the workbook beside this notebook.

In [11]:
MCDS="FOXA1 RB1 RBM28 CDKN1B SMARCB1 ARAF BRCA1 CTCF MKI67 BRCA2 PPP1R9B CCND3 STK11 MPRIP CDH1 AKT2 CHEK2 AKT3 AKT1 HRAS ANXA5 ARID1A RAD51D MSH2 PIK3CA XRN2 TP53 TSPYL5 SMARCD1 CBFB GPS1 XPC GATA3 LTBP1 EGFR MTDH SCUBE2 BRIP1 MYO6 XRCC5 XRCC2 RPA1 RPA2 MLH1 PALB2 ESR1 PLEKHA7 TBX3 SMARCA4 MYBBP1A".split()
drivers=set(pd.read_excel(DATA_FILE,sheet_name='Driver_Genes')['gene_symbol'].dropna().astype(str))&set(G)
N,K=len(G),len(drivers)
if not drivers:
    print('No driver genes found in workbook; driver enrichment and null-model results cannot be reproduced.')
else:
    print('drivers present in network:',K,'(manuscript: 53)')
    cand={'MWDS':MWDS,'Uniform (deg-asc)':da,'MCDS':MCDS,'Centrality top-39 (Average_C, paper)':cent39,'Centrality top-39 (whole-graph recompute)':cent39_global}
    for k,S in cand.items():
        x=len(set(S)&drivers); print('%-42s size %d overlap %d p=%.2e fold=%.2f'%(k,len(S),x,hypergeom.sf(x-1,N,K,len(S)),x/(len(S)*K/N)))
    def fish(a,b):
        xa,xb=len(set(a)&drivers),len(set(b)&drivers); return fisher_exact([[xa,len(a)-xa],[xb,len(b)-xb]])[1]
    print('Fisher MWDS vs centrality %.2f (whole-graph version %.2f), vs uniform %.2f, vs MCDS %.2f'%(fish(MWDS,cent39),fish(MWDS,cent39_global),fish(MWDS,da),fish(MWDS,MCDS)))
    # Degree-matched permutation nulls (10,000 draws, seed 0):
    # Primary null excludes MWDS proteins; conservative null includes them.
    # Within each replicate, sampled proteins are distinct. MWDS vertices are
    # visited in decreasing degree; choose uniformly among available vertices
    # at the same or nearest degree (both sides included for equal-distance ties).
    def degree_matched_null(exclude_mwds, seed=0, reps=10000):
        rng=np.random.default_rng(seed)
        buckets={}
        for u in G:
            if exclude_mwds and u in MWDS:
                continue
            buckets.setdefault(deg[u], []).append(u)
        mwds_order=sorted(MWDS,key=lambda v:-deg[v])
        overlaps=[]
        for _ in range(reps):
            available={d:list(vs) for d,vs in buckets.items()}
            chosen=set()
            for v in mwds_order:
                target=deg[v]; delta=0; nearest=[]
                while not nearest:
                    ds=(target,) if delta==0 else (target-delta,target+delta)
                    for d in ds:
                        if d in available and available[d]:
                            nearest.extend((d,u) for u in available[d])
                    delta+=1
                d,u=nearest[int(rng.integers(len(nearest)))]
                available[d].remove(u)
                chosen.add(u)
            overlaps.append(len(chosen&drivers))
        return np.asarray(overlaps)

    x=len(set(MWDS)&drivers)
    for label,exclude in [('MWDS-excluded',True),('inclusive conservative',False)]:
        ov=degree_matched_null(exclude,seed=0,reps=10000)
        p_emp=(1+(ov>=x).sum())/10001
        print('%s null: mean %.1f sd %.1f 95th pct %.0f max %d; empirical p=%.4f'%(
            label,ov.mean(),ov.std(),np.percentile(ov,95),ov.max(),p_emp))


drivers present in network: 53 (manuscript: 53)
MWDS                                       size 39 overlap 22 p=2.35e-14 fold=5.56
Uniform (deg-asc)                          size 39 overlap 22 p=2.35e-14 fold=5.56
MCDS                                       size 50 overlap 24 p=1.30e-13 fold=4.73
Centrality top-39 (Average_C, paper)       size 39 overlap 19 p=8.34e-11 fold=4.80
Centrality top-39 (whole-graph recompute)  size 39 overlap 18 p=9.79e-10 fold=4.55
Fisher MWDS vs centrality 0.65 (whole-graph version 0.50), vs uniform 1.00, vs MCDS 0.52


MWDS-excluded null: mean 3.5 sd 1.3 95th pct 6 max 9; empirical p=0.0001


inclusive conservative null: mean 10.7 sd 2.1 95th pct 14 max 19; empirical p=0.0001


# Part B. Dog vaccination example (synthetic, illustrative data)
The ten-dog sample below was constructed by the authors; it is **not** municipal or field data.

In [12]:
dw={'Dog1':0.0888889,'Dog2':0.873216,'Dog3':0.428713,'Dog4':1.0,'Dog5':0.0352632,'Dog6':0.654912,'Dog7':0.525497,'Dog8':0.786959,'Dog9':0.185673,'Dog10':0.393509}
raw_h=[1000,200,750,50,900,400,600,150,850,300]; raw_b=[5,1,3,0,4,2,3,1,5,2]; raw_r=[0.8,0.1,0.5,0.0,0.9,0.3,0.4,0.2,0.7,0.6]
nm=lambda x:[(max(x)-v)/(max(x)-min(x)) for v in x]
calc=[0.05*a+0.15*b+0.8*c for a,b,c in zip(nm(raw_h),nm(raw_b),nm(raw_r))]
print('max |recomputed - Table 1| =',max(abs(c-dw[f'Dog{i+1}']) for i,c in enumerate(calc)))
edges=[(1,2),(1,3),(2,3),(3,5),(4,6),(4,7),(5,6),(6,7),(7,8),(8,9),(8,10),(9,10)]
D=nx.Graph(); D.add_nodes_from(dw); D.add_edges_from((f'Dog{a}',f'Dog{b}') for a,b in edges)
print('density %.4f'%nx.density(D)); lt=sorted(dw,key=dw.get)[:3]
print('three lightest dogs',lt,'dominate?',dominates(D,lt),'| dominating triples:',sum(dominates(D,c) for c in itertools.combinations(D,3)),'of 120')

max |recomputed - Table 1| = 4.853801169046879e-07
density 0.2667
three lightest dogs ['Dog5', 'Dog1', 'Dog9'] dominate? False | dominating triples: 15 of 120


In [13]:
def schedule(D, dw, m=3):
    """Iterative MWDS-based vertex reduction (Algorithm 1).

    Each day, search for a minimum-weight size-m dominating set in the
    current candidate subgraph. If none exists, repeatedly discard the
    ceil-half of candidates with the largest weights. Discarded candidates
    remain unvaccinated in H and are considered on a later day. If the
    candidate set has at most m vertices and no size-m dominating set,
    vaccinate all candidates (never more than m).
    Ties in pruning are broken by increasing dog number.
    """
    H = D.copy()
    day = 0
    out = []

    while len(H):
        day += 1
        V1 = H.copy()
        best = None

        # Prune candidates until a size-m dominating set exists or |V1| <= m.
        while len(V1) > m:
            for c in itertools.combinations(V1.nodes(), m):
                if dominates(V1, c):
                    total = sum(dw[v] for v in c)
                    if best is None or total < best[0]:
                        best = (total, c)
            if best is not None:
                break

            # Remove the ceil-half of highest-weight candidates.
            ordered = sorted(V1.nodes(), key=lambda v: (-dw[v], int(v[3:])))
            n_remove = (len(V1) + 1) // 2
            V1.remove_nodes_from(ordered[:n_remove])

        # If |V1| <= m, check the only possible size-m set when |V1| == m.
        if best is None and len(V1) == m:
            c = tuple(V1.nodes())
            if dominates(V1, c):
                best = (sum(dw[v] for v in c), c)

        # Explicit fallback: vaccinate the remaining candidate set (size <= m).
        if best is None:
            best = (sum(dw[v] for v in V1.nodes()), tuple(V1.nodes()))

        selected = list(best[1])
        out.append((day, sorted(selected, key=lambda v: int(v[3:])), round(best[0], 4)))
        H.remove_nodes_from(selected)

    return out

schedule_result = schedule(D, dw, m=3)
for r in schedule_result:
    print('Day %d' % r[0], r[1], r[2])

# Reproducibility and hard-capacity checks against Table 2 in the manuscript.
expected = [
    (1, ['Dog1', 'Dog6', 'Dog9'], 0.9295),
    (2, ['Dog3', 'Dog7', 'Dog10'], 1.3477),
    (3, ['Dog5', 'Dog8'], 0.8222),
    (4, ['Dog2', 'Dog4'], 1.8732),
]
assert schedule_result == expected, f'Schedule differs from manuscript table: {schedule_result}'
assert all(len(group) <= 3 for _, group, _ in schedule_result), 'Daily capacity exceeded'
assert sorted(v for _, group, _ in schedule_result for v in group) == sorted(D.nodes()), 'Not every dog vaccinated exactly once'
print('Validation passed: schedule matches manuscript; capacity <= 3 each day; all 10 dogs vaccinated once.')

Day 1 ['Dog1', 'Dog6', 'Dog9'] 0.9295
Day 2 ['Dog3', 'Dog7', 'Dog10'] 1.3477
Day 3 ['Dog5', 'Dog8'] 0.8222
Day 4 ['Dog2', 'Dog4'] 1.8732
Validation passed: schedule matches manuscript; capacity <= 3 each day; all 10 dogs vaccinated once.


In [14]:
# Recreate manuscript Figure 2: four-day schedule on the synthetic dog graph.
import matplotlib.pyplot as plt
os.makedirs('paper',exist_ok=True)
_layout_d=nx.spring_layout(D,seed=17)
fig,axes=plt.subplots(2,2,figsize=(10,8))
for ax,(day,selected,_) in zip(axes.flat,schedule_result):
    nx.draw_networkx_edges(D,_layout_d,ax=ax,width=1.4,edge_color='#777777',alpha=0.8)
    other=[v for v in D if v not in selected]
    nx.draw_networkx_nodes(D,_layout_d,nodelist=other,ax=ax,node_color='#d9d9d9',node_size=650,edgecolors='black',linewidths=0.8)
    nx.draw_networkx_nodes(D,_layout_d,nodelist=selected,ax=ax,node_color='red',node_size=700,edgecolors='black',linewidths=0.8)
    nx.draw_networkx_labels(D,_layout_d,ax=ax,font_size=8)
    ax.set_title(f'Day {day}: {len(selected)} vaccinated (capacity 3)')
    ax.axis('off')
fig.suptitle('Synthetic ten-dog vaccination schedule (selected dogs in red)',fontsize=13)
fig.tight_layout(); fig.savefig('paper/photo-collage.png',dpi=250,bbox_inches='tight'); plt.close(fig)
print('Generated Figure 2: paper/photo-collage.png')


Generated Figure 2: paper/photo-collage.png
